# Dataset Preprocessing

This notebook handles missing values for all CSV files in `data/raw`:
- Numeric columns: fill with mean
- Categorical columns: fill with mode

Cleaned files are saved to `data/processed` with `_cleaned.csv` suffix.

In [32]:
from pathlib import Path
import pandas as pd

In [33]:
raw_dir = Path('../../data/raw')
processed_dir = Path('../../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(raw_dir.glob('*.csv'))

if not csv_files:
    print('No CSV files found in data/raw')
else:
    for csv_path in csv_files:
        print('\n' + '=' * 90)
        print(f'Processing: {csv_path.name}')
        print('=' * 90)

        df = pd.read_csv(csv_path)

        numeric_cols = df.select_dtypes(include=['number']).columns
        categorical_cols = df.select_dtypes(exclude=['number']).columns

        # Fill missing numeric values with mean
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

        # Fill missing categorical values with mode
        for col in categorical_cols:
            mode_value = df[col].mode(dropna=True)
            if not mode_value.empty:
                df[col] = df[col].fillna(mode_value.iloc[0])

        print('Missing values after preprocessing:')
        print(df.isnull().sum())

        output_path = processed_dir / f'{csv_path.stem}_cleaned.csv'
        df.to_csv(output_path, index=False)
        print(f'Saved: {output_path}')


Processing: ckd-dataset-v2.csv
Missing values after preprocessing:
bp (Diastolic)    0
bp limit          0
sg                0
al                0
class             0
rbc               0
su                0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sod               0
sc                0
pot               0
hemo              0
pcv               0
rbcc              0
wbcc              0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
grf               0
stage             0
affected          0
age               0
dtype: int64
Saved: ..\..\data\processed\ckd-dataset-v2_cleaned.csv

Processing: diabetes.csv
Missing values after preprocessing:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age         

In [34]:
# Detect and remove duplicate rows from each loaded dataset
# (Place this logic after loading a dataframe as `df`)

duplicate_count = df.duplicated().sum()
print(f'Duplicate rows found: {duplicate_count}')

df = df.drop_duplicates().reset_index(drop=True)

print(f'Rows after duplicate removal: {len(df)}')

# Optional: remove duplicates based on specific columns only
# df = df.drop_duplicates(subset=['col1', 'col2']).reset_index(drop=True)

Duplicate rows found: 0
Rows after duplicate removal: 920


In [35]:
# Detect and remove outliers using the IQR method
# Assumes `df` is already loaded

numeric_cols = df.select_dtypes(include=['number']).columns

if len(numeric_cols) == 0:
    print('No numeric columns found for IQR-based outlier detection.')
else:
    q1 = df[numeric_cols].quantile(0.25)
    q3 = df[numeric_cols].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Keep rows where all numeric values are within IQR bounds
    outlier_mask = ((df[numeric_cols] < lower_bound) | (df[numeric_cols] > upper_bound)).any(axis=1)

    rows_before = len(df)
    outliers_found = outlier_mask.sum()

    df = df.loc[~outlier_mask].reset_index(drop=True)

    rows_after = len(df)

    print(f'Numeric columns used: {list(numeric_cols)}')
    print(f'Outlier rows detected: {outliers_found}')
    print(f'Rows before: {rows_before}')
    print(f'Rows after: {rows_after}')

Numeric columns used: ['id', 'age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'num']
Outlier rows detected: 501
Rows before: 920
Rows after: 419


## Outlier Removal Notes (IQR Method)

- This method is applied only to numeric columns.
- For each numeric column, compute:
  - Q1 (25th percentile)
  - Q3 (75th percentile)
  - IQR = Q3 - Q1
- Define bounds:
  - Lower bound = Q1 - 1.5 * IQR
  - Upper bound = Q3 + 1.5 * IQR
- A row is treated as an outlier if any numeric value is outside its column bounds.
- Outlier rows are removed and the dataframe index is reset.

Use this method when you want to reduce the effect of extreme values before training models.

In [36]:
# Standardize noisy categorical tokens before label encoding
# Run this cell before the label-encoding cell

from pathlib import Path
import pandas as pd

raw_dir = Path('../../data/raw')
processed_dir = Path('../../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

dataset_files = [
    raw_dir / 'ckd-dataset-v2.csv',
    raw_dir / 'diabetes.csv',
    raw_dir / 'heart_disease_uci.csv',
]

# Tokens that should be treated as missing/noisy
missing_like_tokens = {
    '', '?', 'na', 'n/a', 'none', 'null', 'nan', 'discrete', 'class', 'p', 'meta'
}

for dataset_path in dataset_files:
    print('\n' + '=' * 90)
    print(f'Standardizing: {dataset_path.name}')
    print('=' * 90)

    if not dataset_path.exists():
        print(f'File not found: {dataset_path}')
        continue

    df = pd.read_csv(dataset_path)
    categorical_cols = df.select_dtypes(include=['object', 'category', 'string']).columns

    if len(categorical_cols) == 0:
        print('No categorical columns found for token standardization.')
    else:
        changed_count = 0

        for col in categorical_cols:
            original = df[col].copy()

            def normalize_token(value):
                if pd.isna(value):
                    return pd.NA
                text = str(value).strip()
                lowered = text.lower()

                if lowered in missing_like_tokens:
                    return pd.NA
                return text

            df[col] = df[col].map(normalize_token)
            changed_count += (original.astype('string') != df[col].astype('string')).sum()

        print(f'Columns standardized: {list(categorical_cols)}')
        print(f'Value updates applied: {int(changed_count)}')

    output_path = processed_dir / f"{dataset_path.stem}_token_standardized.csv"
    df.to_csv(output_path, index=False)
    print(f'Saved standardized file: {output_path}')


Standardizing: ckd-dataset-v2.csv
Columns standardized: ['bp (Diastolic)', 'bp limit', 'sg', 'al', 'class', 'rbc', 'su', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'grf', 'stage', 'affected', 'age']
Value updates applied: 0
Saved standardized file: ..\..\data\processed\ckd-dataset-v2_token_standardized.csv

Standardizing: diabetes.csv
No categorical columns found for token standardization.
Saved standardized file: ..\..\data\processed\diabetes_token_standardized.csv

Standardizing: heart_disease_uci.csv
Columns standardized: ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
Value updates applied: 0
Saved standardized file: ..\..\data\processed\heart_disease_uci_token_standardized.csv


## Token Standardization Notes

- Purpose: clean inconsistent categorical text values before encoding.
- Applied to: `ckd-dataset-v2`, `diabetes`, and `heart_disease_uci`.

### What This Step Does

- Trims leading/trailing whitespace from categorical values.
- Converts noisy placeholder tokens to missing values (`NA`).
- Current missing-like tokens include:
  - `''`, `?`, `na`, `n/a`, `none`, `null`, `nan`
  - `discrete`, `class`, `p`, `meta`

### Why It Matters

- Prevents noisy tokens from becoming real categories during label encoding.
- Improves consistency and model input quality.
- Helps avoid misleading mappings in encoded outputs.

### Output

- Saves cleaned datasets as `data/processed/*_token_standardized.csv`.
- These files are used as preferred inputs in the label-encoding step.

In [37]:
# Convert categorical columns to numeric using Label Encoding
# Uses token-standardized files when available, then saves encoded files

from pathlib import Path
from sklearn.preprocessing import LabelEncoder

raw_dir = Path('../../data/raw')
processed_dir = Path('../../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

dataset_stems = [
    'ckd-dataset-v2',
    'diabetes',
    'heart_disease_uci',
]

for stem in dataset_stems:
    standardized_path = processed_dir / f'{stem}_token_standardized.csv'
    raw_path = raw_dir / f'{stem}.csv'

    # Prefer standardized input so noisy tokens are cleaned before encoding
    input_path = standardized_path if standardized_path.exists() else raw_path

    print('\n' + '=' * 90)
    print(f'Dataset: {input_path.name}')
    print('=' * 90)

    if not input_path.exists():
        print(f'File not found: {input_path}')
        continue

    df = pd.read_csv(input_path)

    # Include string dtype for pandas 2/3 compatibility
    categorical_cols = df.select_dtypes(include=['object', 'category', 'string']).columns

    if len(categorical_cols) == 0:
        print('No categorical columns found for label encoding.')
    else:
        label_encoders = {}

        for col in categorical_cols:
            le = LabelEncoder()
            non_null_mask = df[col].notna()

            # Use nullable integer dtype so encoded ints and missing values coexist safely
            encoded_col = pd.Series(pd.NA, index=df.index, dtype='Int64')
            encoded_col.loc[non_null_mask] = le.fit_transform(df.loc[non_null_mask, col].astype(str))

            df[col] = encoded_col
            label_encoders[col] = le

        print('Label-encoded columns:')
        print(list(categorical_cols))

        print('\nSample mappings (first up to 10 classes per column):')
        for col, le in label_encoders.items():
            classes = list(le.classes_)
            preview = classes[:10]
            mapping_preview = {cls: idx for idx, cls in enumerate(preview)}
            print(f"{col}: {mapping_preview}")

    encoded_output_path = processed_dir / f'{stem}_encoded.csv'
    df.to_csv(encoded_output_path, index=False)
    print(f'\nEncoded file saved to: {encoded_output_path}')


Dataset: ckd-dataset-v2_token_standardized.csv
Label-encoded columns:
['sg', 'al', 'class', 'su', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'grf', 'stage', 'age']

Sample mappings (first up to 10 classes per column):
sg: {'1.009 - 1.011': 0, '1.015 - 1.017': 1, '1.019 - 1.021': 2, '< 1.007': 3, '≥ 1.023': 4}
al: {'1 - 1': 0, '2 - 2': 1, '3 - 3': 2, '< 0': 3, '≥ 4': 4}
class: {'ckd': 0, 'notckd': 1}
su: {'1 - 2': 0, '2 - 2': 1, '3 - 4': 2, '4 - 4': 3, '< 0': 4, '≥ 4': 5}
bgr: {'112 - 154': 0, '154 - 196': 1, '196 - 238': 2, '238 - 280': 3, '280 - 322': 4, '322 - 364': 5, '364 - 406': 6, '406 - 448': 7, '< 112': 8, '≥ 448': 9}
bu: {'124.3 - 162.4': 0, '162.4 - 200.5': 1, '200.5 - 238.6': 2, '238.6 - 276.7': 3, '48.1 - 86.2': 4, '86.2 - 124.3': 5, '< 48.1': 6, '≥ 352.9': 7}
sod: {'118 - 123': 0, '123 - 128': 1, '128 - 133': 2, '133 - 138': 3, '138 - 143': 4, '143 - 148': 5, '148 - 153': 6, '< 118': 7, '≥ 158': 8}
sc: {'13.1 - 16.25': 0, '16.25 - 19.4': 1, '3.65 - 6.

## Label Encoding Notes

- Purpose: convert categorical text columns into numeric values required by most ML models.
- Applied datasets: `ckd-dataset-v2`, `diabetes`, and `heart_disease_uci`.

### Updated Pipeline

- Step 1: run token standardization first.
  - Creates `*_token_standardized.csv` files in `data/processed`.
  - Treats noisy placeholders as missing values (for example `discrete`, `class`, `p`, `meta`, `?`, `na`).
- Step 2: run label encoding.
  - Reads token-standardized files when available (falls back to raw files only if needed).
  - Encodes categorical columns with integer labels.
  - Preserves missing values as missing (`NA`) using nullable integer dtype.

### Output Files

- Standardized: `data/processed/*_token_standardized.csv`
- Encoded: `data/processed/*_encoded.csv`

### Modeling Caution

- Label encoding introduces arbitrary numeric order (0, 1, 2, ...).
- Tree-based models usually handle this well.
- For linear, distance-based, or neural models, consider one-hot encoding for nominal features.

In [38]:
# Feature scaling using StandardScaler for all available datasets

from sklearn.preprocessing import StandardScaler

# Prefer cleaned_splits (target-missing rows already removed)
source_splits = cleaned_splits if 'cleaned_splits' in locals() and cleaned_splits else splits

if 'source_splits' not in locals() or not source_splits:
    print("No dataset splits available. Run split/cleanup cells first.")
else:
    scaled_splits = {}

    for dataset_name, split_data in source_splits.items():
        X = split_data['X']
        y = split_data['y']

        # Skip ID-like columns from scaling based on column-name heuristics
        id_like_cols = [
            col for col in X.columns
            if str(col).strip().lower() == 'id' or str(col).strip().lower().endswith('_id')
        ]

        cols_to_scale = [col for col in X.columns if col not in id_like_cols]

        if not cols_to_scale:
            X_scaled = X.copy()
            scaler = None
        else:
            scaler = StandardScaler()
            X_scaled_array = scaler.fit_transform(X[cols_to_scale])
            X_scaled_part = pd.DataFrame(X_scaled_array, columns=cols_to_scale, index=X.index)

            # Rebuild dataframe to avoid dtype assignment conflicts on integer columns
            if id_like_cols:
                X_scaled = pd.concat([X_scaled_part, X[id_like_cols]], axis=1)[X.columns]
            else:
                X_scaled = X_scaled_part

        # Store scaled outputs for downstream training
        scaled_splits[dataset_name] = {
            'X': X_scaled,
            'y': y,
            'scaler': scaler,
            'scaled_columns': cols_to_scale,
            'excluded_columns': id_like_cols,
        }

        print('\n' + '=' * 90)
        print(f'Scaled dataset: {dataset_name}')
        print('=' * 90)
        print('Original X shape:', X.shape)
        print('Scaled X shape:', X_scaled.shape)
        print('Excluded from scaling (ID-like):', id_like_cols if id_like_cols else 'None')
        print('Scaled column count:', len(cols_to_scale))

        if cols_to_scale:
            print('Scaled feature means (first 5 of scaled columns):')
            print(X_scaled[cols_to_scale].mean().head())
            print('Scaled feature std (first 5 of scaled columns):')
            print(X_scaled[cols_to_scale].std(ddof=0).head())
        else:
            print('No columns were scaled for this dataset.')

    print('\nCompleted scaling for datasets:')
    print(list(scaled_splits.keys()))


Scaled dataset: diabetes
Original X shape: (768, 8)
Scaled X shape: (768, 8)
Excluded from scaling (ID-like): None
Scaled column count: 8
Scaled feature means (first 5 of scaled columns):
Pregnancies     -6.476301e-17
Glucose         -9.251859e-18
BloodPressure    1.503427e-17
SkinThickness    1.006140e-16
Insulin         -3.006854e-17
dtype: float64
Scaled feature std (first 5 of scaled columns):
Pregnancies      1.0
Glucose          1.0
BloodPressure    1.0
SkinThickness    1.0
Insulin          1.0
dtype: float64

Scaled dataset: heart
Original X shape: (920, 15)
Scaled X shape: (920, 15)
Excluded from scaling (ID-like): ['id']
Scaled column count: 14
Scaled feature means (first 5 of scaled columns):
age         6.178632e-17
sex         6.178632e-17
dataset    -1.235726e-16
cp         -4.633974e-17
trestbps   -7.014650e-16
dtype: float64
Scaled feature std (first 5 of scaled columns):
age         1.0
sex         1.0
dataset     1.0
cp          1.0
trestbps    1.0
dtype: float64

Sca

## Feature Scaling Notes

- Purpose: standardize feature columns across all prepared datasets for consistent model input scale.
- Scope: loops through every dataset available in cleaned splits (or split data if cleaned splits are not available).

### What The Scaling Cell Does

- Uses StandardScaler independently for each dataset.
- Detects ID-like columns and excludes them from scaling:
  - column name equal to id
  - column name ending with _id
- Scales only non-ID columns, then rebuilds the dataset in original column order.
- Preserves target values without modification.


### Stored Outputs

- Results are saved in scaled_splits for each dataset.
- Each dataset entry contains:
  - X: scaled feature dataframe
  - y: target series
  - scaler: fitted StandardScaler object (None if no columns were eligible)
  - scaled_columns: list of columns that were scaled
  - excluded_columns: list of ID-like columns skipped from scaling

### Validation Signals

- Original and scaled shapes should match.
- Means of scaled columns should be approximately 0.
- Standard deviation of scaled columns should be approximately 1 (using population std).

In [39]:
# Split all datasets into features (X) and target (y)

from pathlib import Path
import pandas as pd

processed_dir = Path('../../data/processed')

file_map = {
    'diabetes': processed_dir / 'diabetes_encoded.csv',
    'heart': processed_dir / 'heart_disease_uci_encoded.csv',
    'kidney': processed_dir / 'ckd-dataset-v2_encoded.csv',
}

target_map = {
    'diabetes': 'Outcome',
    'heart': 'num',
    'kidney': 'class',
}

# Store splits for downstream modeling
splits = {}

for dataset_name, input_file in file_map.items():
    print('\n' + '=' * 90)
    print(f'Dataset: {dataset_name}')
    print('=' * 90)

    if not input_file.exists():
        print(f'File not found: {input_file}')
        continue

    df = pd.read_csv(input_file)
    target_col = target_map[dataset_name]

    if target_col not in df.columns:
        print(f"Mapped target '{target_col}' was not found in the dataframe.")
        print('Available columns:')
        print(df.columns.tolist())
        continue

    X = df.drop(columns=[target_col])
    y = df[target_col]
    splits[dataset_name] = {'X': X, 'y': y}

    print(f'Input file: {input_file.name}')
    print(f'Selected target column: {target_col}')
    print('Feature matrix shape (X):', X.shape)
    print('Target vector shape (y):', y.shape)
    print('Target distribution:')
    print(y.value_counts(dropna=False))

print('\nCompleted splits for datasets:')
print(list(splits.keys()))


Dataset: diabetes
Input file: diabetes_encoded.csv
Selected target column: Outcome
Feature matrix shape (X): (768, 8)
Target vector shape (y): (768,)
Target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Dataset: heart
Input file: heart_disease_uci_encoded.csv
Selected target column: num
Feature matrix shape (X): (920, 15)
Target vector shape (y): (920,)
Target distribution:
num
0    411
1    265
2    109
3    107
4     28
Name: count, dtype: int64

Dataset: kidney
Input file: ckd-dataset-v2_encoded.csv
Selected target column: class
Feature matrix shape (X): (202, 28)
Target vector shape (y): (202,)
Target distribution:
class
0.0    128
1.0     72
NaN      2
Name: count, dtype: int64

Completed splits for datasets:
['diabetes', 'heart', 'kidney']


In [40]:
# Drop rows with missing target values for each dataset split

if 'splits' not in locals() or not splits:
    print("No splits found. Run the previous split cell first.")
else:
    cleaned_splits = {}

    for dataset_name, split_data in splits.items():
        X = split_data['X']
        y = split_data['y']

        before_rows = len(y)
        valid_mask = y.notna()

        X_clean = X.loc[valid_mask].reset_index(drop=True)
        y_clean = y.loc[valid_mask].reset_index(drop=True)

        after_rows = len(y_clean)
        dropped_rows = before_rows - after_rows

        cleaned_splits[dataset_name] = {'X': X_clean, 'y': y_clean}

        print('\n' + '=' * 90)
        print(f'Dataset: {dataset_name}')
        print('=' * 90)
        print(f'Rows before target cleanup: {before_rows}')
        print(f'Rows after target cleanup:  {after_rows}')
        print(f'Rows dropped:               {dropped_rows}')
        print('Clean target distribution:')
        print(y_clean.value_counts(dropna=False))

    print('\nCleaned splits available in variable: cleaned_splits')


Dataset: diabetes
Rows before target cleanup: 768
Rows after target cleanup:  768
Rows dropped:               0
Clean target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Dataset: heart
Rows before target cleanup: 920
Rows after target cleanup:  920
Rows dropped:               0
Clean target distribution:
num
0    411
1    265
2    109
3    107
4     28
Name: count, dtype: int64

Dataset: kidney
Rows before target cleanup: 202
Rows after target cleanup:  200
Rows dropped:               2
Clean target distribution:
class
0.0    128
1.0     72
Name: count, dtype: int64

Cleaned splits available in variable: cleaned_splits


In [41]:
# Diagnose kidney rows dropped due to missing target values

if 'splits' not in locals() or 'kidney' not in splits:
    print("Kidney split not found. Run the split cell first.")
else:
    X_kidney = splits['kidney']['X']
    y_kidney = splits['kidney']['y']

    missing_target_mask = y_kidney.isna()
    dropped_count = int(missing_target_mask.sum())

    print(f'Kidney rows with missing target (class): {dropped_count}')

    if dropped_count == 0:
        print('No kidney rows will be dropped by target cleanup.')
    else:
        dropped_rows_df = X_kidney.loc[missing_target_mask].copy()
        dropped_rows_df.insert(0, 'class', y_kidney.loc[missing_target_mask].values)

        print('Dropped row indices in kidney split:')
        print(list(y_kidney.index[missing_target_mask]))

        print('\nDropped kidney rows preview:')
        print(dropped_rows_df.head())

Kidney rows with missing target (class): 2
Dropped row indices in kidney split:
[0, 1]

Dropped kidney rows preview:
   class  bp (Diastolic)  bp limit  sg  al  rbc  su  pc  pcc  ba  ...  htn  \
0    NaN             NaN       NaN NaN NaN  NaN NaN NaN  NaN NaN  ...  NaN   
1    NaN             NaN       NaN NaN NaN  NaN NaN NaN  NaN NaN  ...  NaN   

   dm  cad  appet  pe  ane  grf  stage  affected  age  
0 NaN  NaN    NaN NaN  NaN  NaN    NaN       NaN  NaN  
1 NaN  NaN    NaN NaN  NaN  NaN    NaN       NaN  NaN  

[2 rows x 29 columns]


## Feature and Target Splitting Notes

- Purpose: split each encoded dataset into:
  - `X` (feature matrix)
  - `y` (target vector)

### Dataset-to-Target Mapping

- Diabetes -> `Outcome`
- Heart -> `num`
- Kidney -> `class`

### How Splitting Works

- Encoded files are loaded from `data/processed`.
- For each dataset:
  - `X = df.drop(columns=[target_col])`
  - `y = df[target_col]`
- Results are stored in `splits`:
  - `splits['diabetes']['X']`, `splits['diabetes']['y']`
  - `splits['heart']['X']`, `splits['heart']['y']`
  - `splits['kidney']['X']`, `splits['kidney']['y']`

### Missing Target Cleanup

- A follow-up cleanup step removes rows where target `y` is missing.
- Cleaned outputs are stored in `cleaned_splits` with the same key structure.
- Use `cleaned_splits` for model training to avoid invalid labels.